# S6E9 · V11 FINAL — Internal + Optional External OOF Garage

这是最终整合版。它**不重新训练任何大模型**，直接复用你已经跑完的：

- CatBoost
- LightGBM
- old XGBoost
- V8 strong_rich_te
- V8 strong_digit_freq
- V10 window XGBoost

然后做：

1. ID 对齐
2. 单模型 OOF AUC
3. Spearman 去重
4. rank → probit
5. 5-fold Nested Caruana Hill Climb
6. 与内部 V8 reference 做 fold-by-fold gate
7. 可选扫描 `external_oof/` 中的外部 OOF/test 预测
8. 输出 OOF-only final 与 rules experimental 两个 submission

默认项目目录：

`notebooks/` 的上一级目录

目标列：

`Will_Buy_EV`

这份 notebook 的目标是：**快速给出最终可提交版本，而不是继续训练新模型。**

In [1]:
from pathlib import Path
from datetime import datetime
import json, re, warnings
import numpy as np
import pandas as pd

from scipy.stats import rankdata, norm
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings("ignore")

PROJECT = Path.cwd().parent  # 项目根目录（notebooks/ 的上一级）
TARGET = "Will_Buy_EV"
ID_COL = "id"

META_FOLDS = 5
META_SEED = 2026

HILL_MAX_STEPS = 120
HILL_TOL = 1e-7

# Hill 只考虑不太弱的 engine。
MIN_ENGINE_AUC = 0.9450

# 近乎重复的 engine 去掉，避免 garage 被同一种预测淹没。
MAX_ABS_SPEARMAN_DUP = 0.99995

# 可选外部预测目录。没有也能完整运行。
EXTERNAL_DIR = PROJECT / "external_oof"

OUT_DIR = PROJECT / "outputs" / (
    "v11_final_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
)
OUT_DIR.mkdir(parents=True, exist_ok=False)

print("PROJECT:", PROJECT)
print("OUT_DIR:", OUT_DIR)
print("EXTERNAL_DIR:", EXTERNAL_DIR)

PROJECT: /Users/yw/unsw/Kaggle
OUT_DIR: /Users/yw/unsw/Kaggle/outputs/v11_final_20261001_031756_836748
EXTERNAL_DIR: /Users/yw/unsw/Kaggle/external_oof


## 1. 加载数据 + 通用读取函数

所有 OOF / test prediction 都按 `id` 显式对齐，避免 CSV 行顺序造成隐蔽错误。

In [2]:
def require(condition, message):
    if not bool(condition):
        raise ValueError(message)

def load_data(project):
    train = pd.read_csv(project / "data/train.csv")
    test = pd.read_csv(project / "data/test.csv")
    sample_path = project / "data/sample_submission.csv"
    sample = pd.read_csv(sample_path) if sample_path.is_file() else None

    require(TARGET in train.columns, f"Missing target: {TARGET}")
    require(train[ID_COL].is_unique, "Duplicate train ids")
    require(test[ID_COL].is_unique, "Duplicate test ids")

    y = train[TARGET].map({"No": 0, "Yes": 1})
    require(y.notna().all(), "Expected No/Yes target")
    y = y.astype(np.int8)

    return train, test, sample, y

train, test, sample, y = load_data(PROJECT)

def align_pred(path, ids, pred_col):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)

    frame = pd.read_csv(path)
    require(ID_COL in frame.columns, f"{path.name}: missing id")
    require(pred_col in frame.columns, f"{path.name}: missing {pred_col}")
    require(frame[ID_COL].is_unique, f"{path.name}: duplicate ids")

    wanted = pd.Index(np.asarray(ids))
    require(len(frame) == len(wanted), f"{path.name}: wrong row count")
    require(wanted.isin(frame[ID_COL]).all(), f"{path.name}: id mismatch")

    values = (
        frame.set_index(ID_COL)
        .loc[wanted, pred_col]
        .to_numpy(dtype=np.float64)
    )
    require(np.isfinite(values).all(), f"{path.name}: non-finite prediction")
    return values

def rank01(values):
    values = np.asarray(values, dtype=np.float64)
    require(np.isfinite(values).all(), "rank01 got non-finite values")
    return rankdata(values, method="average") / len(values)

def probit_rank(values):
    values = np.asarray(values, dtype=np.float64)
    require(np.isfinite(values).all(), "probit_rank got non-finite values")
    r = rankdata(values, method="average")
    return norm.ppf(r / (len(values) + 1.0)).astype(np.float64)

def save_submission(values, path):
    values = rank01(values)

    sub = pd.DataFrame({
        ID_COL: test[ID_COL].to_numpy(),
        TARGET: values,
    })

    if sample is not None:
        sub = (
            sample[[ID_COL]]
            .merge(
                sub,
                on=ID_COL,
                how="left",
                validate="one_to_one",
                sort=False,
            )
        )

    require(sub[TARGET].notna().all(), f"{path.name}: missing predictions")
    sub.to_csv(path, index=False)

print("train:", train.shape)
print("test :", test.shape)
print("positive rate:", float(y.mean()))

train: (668665, 15)
test : (286571, 14)
positive rate: 0.17464500160768098


## 2. 自动寻找并加载你自己的预测

下面会自动寻找最新的完整 V8 / V10 目录。

In [3]:
CAT_DIR = PROJECT / "outputs/v3_20260925_180457_278867"
LGB_DIR = PROJECT / "outputs/v4_lightgbm_20260930_163526_079778"
OLD_XGB_DIR = PROJECT / "outputs/v5_xgb10_20260930_205349_440980"

def newest_complete(pattern, required_files):
    dirs = sorted(
        (PROJECT / "outputs").glob(pattern),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    for d in dirs:
        if all((d / f).is_file() for f in required_files):
            return d
    raise FileNotFoundError(
        f"No complete directory for {pattern}; required={required_files}"
    )

V8_STRONG_DIR = newest_complete(
    "v8_strong_xgb_*",
    [
        "oof_strong_rich_te.csv",
        "submission_strong_rich_te.csv",
        "oof_strong_digit_freq.csv",
        "submission_strong_digit_freq.csv",
    ],
)

V10_DIR = newest_complete(
    "v10_window_xgb_*",
    [
        "oof_window_xgb.csv",
        "submission_window_xgb.csv",
    ],
)

def internal_pair(run_dir, oof_file, test_file, oof_col="prediction", test_col=TARGET):
    return {
        "oof": align_pred(run_dir / oof_file, train[ID_COL], oof_col),
        "test": align_pred(run_dir / test_file, test[ID_COL], test_col),
    }

cat = internal_pair(CAT_DIR, "oof_catboost.csv", "submission_catboost.csv")
lgb = internal_pair(LGB_DIR, "oof_lightgbm.csv", "submission_lightgbm.csv")
old_xgb = internal_pair(OLD_XGB_DIR, "oof_xgboost.csv", "submission_xgboost.csv")

strong_rich = internal_pair(
    V8_STRONG_DIR,
    "oof_strong_rich_te.csv",
    "submission_strong_rich_te.csv",
)

strong_digit = internal_pair(
    V8_STRONG_DIR,
    "oof_strong_digit_freq.csv",
    "submission_strong_digit_freq.csv",
)

window_xgb = internal_pair(
    V10_DIR,
    "oof_window_xgb.csv",
    "submission_window_xgb.csv",
)

print("V8_STRONG_DIR:", V8_STRONG_DIR)
print("V10_DIR:", V10_DIR)

V8_STRONG_DIR: /Users/yw/unsw/Kaggle/outputs/v8_strong_xgb_20260930_213758_937864
V10_DIR: /Users/yw/unsw/Kaggle/outputs/v10_window_xgb_20261001_030543_973913


## 3. 构造内部 garage

除了基础模型，还加入少量已经验证过的透明 blend view。

这里不加入 RealMLP，因为你本地 RealMLP OOF 只有约 `0.939`，明显太弱。

In [4]:
current_oof = 0.5 * cat["oof"] + 0.5 * lgb["oof"]
current_test = 0.5 * cat["test"] + 0.5 * lgb["test"]

v5_oof = 0.5 * rank01(old_xgb["oof"]) + 0.5 * rank01(current_oof)
v5_test = 0.5 * rank01(old_xgb["test"]) + 0.5 * rank01(current_test)

internal_oof = {
    "catboost": cat["oof"],
    "lightgbm": lgb["oof"],
    "old_xgb": old_xgb["oof"],
    "current_cat_lgb": current_oof,
    "v5_baseline": v5_oof,

    "strong_rich_te": strong_rich["oof"],
    "strong_rich_current_rank50":
        0.5 * rank01(strong_rich["oof"])
        + 0.5 * rank01(current_oof),

    "strong_digit_freq": strong_digit["oof"],
    "strong_digit_current_rank50":
        0.5 * rank01(strong_digit["oof"])
        + 0.5 * rank01(current_oof),

    "v8_views_equal_rank":
        0.5 * rank01(strong_rich["oof"])
        + 0.5 * rank01(strong_digit["oof"]),

    "window_xgb": window_xgb["oof"],
    "window_v5_rank10":
        0.10 * rank01(window_xgb["oof"])
        + 0.90 * rank01(v5_oof),
}

internal_test = {
    "catboost": cat["test"],
    "lightgbm": lgb["test"],
    "old_xgb": old_xgb["test"],
    "current_cat_lgb": current_test,
    "v5_baseline": v5_test,

    "strong_rich_te": strong_rich["test"],
    "strong_rich_current_rank50":
        0.5 * rank01(strong_rich["test"])
        + 0.5 * rank01(current_test),

    "strong_digit_freq": strong_digit["test"],
    "strong_digit_current_rank50":
        0.5 * rank01(strong_digit["test"])
        + 0.5 * rank01(current_test),

    "v8_views_equal_rank":
        0.5 * rank01(strong_rich["test"])
        + 0.5 * rank01(strong_digit["test"]),

    "window_xgb": window_xgb["test"],
    "window_v5_rank10":
        0.10 * rank01(window_xgb["test"])
        + 0.90 * rank01(v5_test),
}

internal_auc = (
    pd.DataFrame([
        {"engine": n, "oof_auc": roc_auc_score(y, p)}
        for n, p in internal_oof.items()
    ])
    .sort_values("oof_auc", ascending=False)
    .reset_index(drop=True)
)

print(internal_auc.to_string(index=False, float_format=lambda v: f"{v:.9f}"))

                     engine     oof_auc
 strong_rich_current_rank50 0.946112963
           window_v5_rank10 0.946109184
                v5_baseline 0.946098745
             strong_rich_te 0.945998429
            current_cat_lgb 0.945986243
                   lightgbm 0.945946381
                    old_xgb 0.945931543
        v8_views_equal_rank 0.945759391
strong_digit_current_rank50 0.945620865
                   catboost 0.945618233
                 window_xgb 0.945516020
          strong_digit_freq 0.944402544


## 4. 可选：自动扫描 `external_oof/`

如果你明天下载任何公开 OOF/test 文件，只要放到：

项目根目录下的 `external_oof/`

并使用相似文件名，例如：

- `blamerx_oof.csv`
- `blamerx_test.csv`

或：

- `realmlp_oof.csv`
- `realmlp_submission.csv`

这个 cell 会尽量自动配对。

没有外部文件时会直接跳过，最终版本仍可完整运行。

In [5]:
PRED_COL_PREFERENCE = [
    "prediction",
    "pred",
    "OOF_Pred",
    "oof_pred",
    "test_pred",
    "oof",
    TARGET,
]

def normalized_file_key(path):
    stem = path.stem.lower()
    for token in [
        "submission", "predictions", "prediction",
        "preds", "oof", "test", "train"
    ]:
        stem = re.sub(rf"(^|[_\-\s]){token}([_\-\s]|$)", "_", stem)
    stem = re.sub(r"[_\-\s]+", "_", stem).strip("_")
    return stem

def choose_pred_col(frame, is_oof):
    for col in PRED_COL_PREFERENCE:
        if col in frame.columns:
            # In an OOF file TARGET may be the true label rather than prediction.
            if is_oof and col == TARGET and "prediction" in frame.columns:
                continue
            return col

    numeric = [
        c for c in frame.columns
        if c != ID_COL and pd.api.types.is_numeric_dtype(frame[c])
    ]
    # Avoid common true-label columns.
    numeric = [c for c in numeric if c.lower() not in {"target", "y", "y_true", "label"}]
    if len(numeric) == 1:
        return numeric[0]

    raise ValueError(f"Cannot infer prediction column from {frame.columns.tolist()}")

def scan_external_pairs(folder):
    folder = Path(folder)
    if not folder.exists():
        return {}

    csvs = sorted(folder.glob("*.csv"))
    groups = {}

    for path in csvs:
        key = normalized_file_key(path)
        groups.setdefault(key, []).append(path)

    pairs = {}

    for key, files in groups.items():
        oof_candidates = [
            p for p in files
            if ("oof" in p.stem.lower() or "train" in p.stem.lower())
        ]
        test_candidates = [
            p for p in files
            if ("test" in p.stem.lower() or "submission" in p.stem.lower())
        ]

        if not oof_candidates or not test_candidates:
            continue

        oof_path = oof_candidates[0]
        test_path = test_candidates[0]

        try:
            oof_frame = pd.read_csv(oof_path)
            test_frame = pd.read_csv(test_path)

            oof_col = choose_pred_col(oof_frame, is_oof=True)
            test_col = choose_pred_col(test_frame, is_oof=False)

            oof_values = align_pred(oof_path, train[ID_COL], oof_col)
            test_values = align_pred(test_path, test[ID_COL], test_col)

            pairs[f"ext_{key}"] = {
                "oof": oof_values,
                "test": test_values,
                "oof_file": str(oof_path),
                "test_file": str(test_path),
                "oof_col": oof_col,
                "test_col": test_col,
            }

        except Exception as exc:
            print(f"Skipped external group {key}: {exc}")

    return pairs

external_pairs = scan_external_pairs(EXTERNAL_DIR)

print("External engines found:", len(external_pairs))
for name, info in external_pairs.items():
    print(name, "|", info["oof_file"], "|", info["test_file"])

External engines found: 0


## 5. 先重建纯内部 V8+V10 reference

这个 baseline 是最终 expanded garage 的比较对象。

In [6]:
def make_matrix(oof_dict, test_dict, names):
    P_oof = np.column_stack([probit_rank(oof_dict[n]) for n in names])
    P_test = np.column_stack([probit_rank(test_dict[n]) for n in names])
    return P_oof, P_test

def hill_select(P, labels, candidates):
    blend = np.zeros(len(labels), dtype=np.float64)
    counts = np.zeros(P.shape[1], dtype=np.float64)
    best = -np.inf

    for _ in range(HILL_MAX_STEPS):
        scores = {
            j: roc_auc_score(labels, blend + P[:, j])
            for j in candidates
        }
        j = max(scores, key=scores.get)
        score = scores[j]

        if score <= best + HILL_TOL:
            break

        blend += P[:, j]
        counts[j] += 1.0
        best = score

    require(counts.sum() > 0, "Hill selected no engines")
    return counts / counts.sum()

def nested_hill(oof_dict, test_dict):
    names = list(oof_dict)
    P_oof, P_test = make_matrix(oof_dict, test_dict, names)

    single_auc = np.array([
        roc_auc_score(y, oof_dict[n])
        for n in names
    ])

    candidates = [
        i for i, auc in enumerate(single_auc)
        if auc >= MIN_ENGINE_AUC
    ]
    require(candidates, "No candidates pass MIN_ENGINE_AUC")

    folds = list(
        StratifiedKFold(
            META_FOLDS,
            shuffle=True,
            random_state=META_SEED,
        ).split(P_oof, y)
    )

    nested_oof = np.zeros(len(y), dtype=np.float64)
    fold_rows = []

    for fold, (tr, va) in enumerate(folds, start=1):
        w = hill_select(P_oof[tr], np.asarray(y)[tr], candidates)
        pred = P_oof[va] @ w
        nested_oof[va] = pred

        fold_rows.append({
            "fold": fold,
            "auc": float(roc_auc_score(np.asarray(y)[va], pred)),
            "engines_used": int((w > 0).sum()),
        })

    w_full = hill_select(P_oof, np.asarray(y), candidates)
    test_pred = P_test @ w_full

    return {
        "names": names,
        "P_oof": P_oof,
        "P_test": P_test,
        "oof": nested_oof,
        "test": test_pred,
        "auc": float(roc_auc_score(y, nested_oof)),
        "folds": folds,
        "fold_table": pd.DataFrame(fold_rows),
        "weights": pd.Series(w_full, index=names),
    }

internal_ref = nested_hill(internal_oof, internal_test)

print("Internal nested-hill OOF:", internal_ref["auc"])
print("\nInternal full-data weights:")
print(
    internal_ref["weights"]
    .loc[lambda s: s > 0]
    .sort_values(ascending=False)
    .to_string(float_format=lambda v: f"{v:.4f}")
)

Internal nested-hill OOF: 0.9461701638092593

Internal full-data weights:
lightgbm                     0.3333
old_xgb                      0.1667
strong_rich_te               0.1667
strong_rich_current_rank50   0.1667
v8_views_equal_rank          0.1667


## 6. 合并外部 engine、做 AUC / correlation 去重

规则：

- OOF AUC < `0.9450` 不进入 hill
- 与已经保留的更强 engine 的绝对 Spearman > `0.99995` 时去重
- 外部文件不会因为 Public 分数高就自动获得权重，只看 OOF

In [7]:
garage_oof = dict(internal_oof)
garage_test = dict(internal_test)

for name, info in external_pairs.items():
    garage_oof[name] = info["oof"]
    garage_test[name] = info["test"]

all_rows = []
for name, pred in garage_oof.items():
    all_rows.append({
        "engine": name,
        "oof_auc": float(roc_auc_score(y, pred)),
    })

all_auc = (
    pd.DataFrame(all_rows)
    .sort_values("oof_auc", ascending=False)
    .reset_index(drop=True)
)

eligible_names = all_auc.loc[
    all_auc["oof_auc"] >= MIN_ENGINE_AUC,
    "engine"
].tolist()

kept = []
dropped = []

for name in eligible_names:
    pred = garage_oof[name]
    duplicate_of = None

    for kept_name in kept:
        corr = float(
            pd.Series(pred).corr(
                pd.Series(garage_oof[kept_name]),
                method="spearman",
            )
        )
        if abs(corr) >= MAX_ABS_SPEARMAN_DUP:
            duplicate_of = kept_name
            break

    if duplicate_of is None:
        kept.append(name)
    else:
        dropped.append({
            "engine": name,
            "duplicate_of": duplicate_of,
            "reason": f"|Spearman| >= {MAX_ABS_SPEARMAN_DUP}",
        })

filtered_oof = {n: garage_oof[n] for n in kept}
filtered_test = {n: garage_test[n] for n in kept}

print("All engines:", len(garage_oof))
print("Eligible:", len(eligible_names))
print("Kept after correlation filter:", len(kept))
print("\nTop engines:")
print(all_auc.head(20).to_string(index=False, float_format=lambda v: f"{v:.9f}"))

if dropped:
    print("\nDropped near-duplicates:")
    print(pd.DataFrame(dropped).to_string(index=False))

All engines: 12
Eligible: 11
Kept after correlation filter: 10

Top engines:
                     engine     oof_auc
 strong_rich_current_rank50 0.946112963
           window_v5_rank10 0.946109184
                v5_baseline 0.946098745
             strong_rich_te 0.945998429
            current_cat_lgb 0.945986243
                   lightgbm 0.945946381
                    old_xgb 0.945931543
        v8_views_equal_rank 0.945759391
strong_digit_current_rank50 0.945620865
                   catboost 0.945618233
                 window_xgb 0.945516020
          strong_digit_freq 0.944402544

Dropped near-duplicates:
     engine     duplicate_of                reason
v5_baseline window_v5_rank10 |Spearman| >= 0.99995


## 7. Expanded Nested Hill + gate

最终只有在：

- expanded overall OOF > internal reference
- 至少 4/5 meta folds 提升
- 平均 fold delta > 0

时才自动采用 expanded garage。

否则退回内部 reference。

In [8]:
expanded = nested_hill(filtered_oof, filtered_test)

fold_compare = []

for fold, ((_, va), internal_row, expanded_row) in enumerate(
    zip(
        internal_ref["folds"],
        internal_ref["fold_table"].to_dict("records"),
        expanded["fold_table"].to_dict("records"),
    ),
    start=1,
):
    internal_auc_fold = float(
        roc_auc_score(np.asarray(y)[va], internal_ref["oof"][va])
    )
    expanded_auc_fold = float(
        roc_auc_score(np.asarray(y)[va], expanded["oof"][va])
    )

    fold_compare.append({
        "fold": fold,
        "internal_auc": internal_auc_fold,
        "expanded_auc": expanded_auc_fold,
        "delta": expanded_auc_fold - internal_auc_fold,
    })

fold_compare = pd.DataFrame(fold_compare)

wins = int((fold_compare["delta"] > 0).sum())
mean_delta = float(fold_compare["delta"].mean())
overall_delta = float(expanded["auc"] - internal_ref["auc"])

expanded_accepted = bool(
    overall_delta > 0
    and wins >= 4
    and mean_delta > 0
)

print("Internal OOF:", internal_ref["auc"])
print("Expanded OOF:", expanded["auc"])
print("Delta:", overall_delta)
print(f"Wins: {wins}/{META_FOLDS}")
print("Mean fold delta:", mean_delta)
print("Expanded accepted:", expanded_accepted)

print("\nFold comparison:")
print(fold_compare.to_string(index=False, float_format=lambda v: f"{v:.9f}"))

print("\nExpanded full-data weights:")
print(
    expanded["weights"]
    .loc[lambda s: s > 0]
    .sort_values(ascending=False)
    .to_string(float_format=lambda v: f"{v:.4f}")
)

Internal OOF: 0.9461701638092593
Expanded OOF: 0.9461701638092593
Delta: 0.0
Wins: 0/5
Mean fold delta: 0.0
Expanded accepted: False

Fold comparison:
 fold  internal_auc  expanded_auc       delta
    1   0.945720823   0.945720823 0.000000000
    2   0.945443547   0.945443547 0.000000000
    3   0.947507143   0.947507143 0.000000000
    4   0.946194891   0.946194891 0.000000000
    5   0.945984859   0.945984859 0.000000000

Expanded full-data weights:
lightgbm                     0.3333
strong_rich_current_rank50   0.1667
strong_rich_te               0.1667
old_xgb                      0.1667
v8_views_equal_rank          0.1667


## 8. Published hard rules（单独保存 experimental）

规则来自社区对完整训练数据的观察，所以不把它与严格 nested OOF 的证据混为一谈。

最终会同时保存：

- `submission_final_oof_only.csv`
- `submission_final_rules_experimental.csv`

In [9]:
def rule_masks(frame):
    income = pd.to_numeric(frame["Annual_Income_USD"], errors="coerce").to_numpy()
    commute = pd.to_numeric(frame["Daily_Commute_km"], errors="coerce").to_numpy()

    subsidy_no = (
        frame["Subsidy_Available"]
        .astype(str)
        .str.strip()
        .str.lower()
        .to_numpy()
        == "no"
    )

    concern1 = (
        pd.to_numeric(
            frame["Environmental_Concern_Level"],
            errors="coerce",
        ).to_numpy()
        == 1
    )

    anxious = np.isin(
        frame["Range_Anxiety_Level"]
        .astype(str)
        .str.strip()
        .to_numpy(),
        ["Medium", "High"],
    )

    return [
        ("income >= 170537", income >= 170537, +1),
        ("income 31004-41970", (income >= 31004) & (income <= 41970), -1),
        ("commute >= 83", commute >= 83, -1),
        (
            "30k no subsidy concern1/anxious",
            (income == 30000) & subsidy_no & (concern1 | anxious),
            -1,
        ),
    ]

def apply_rules(values, frame):
    r = rank01(values)

    for _, mask, sign in rule_masks(frame):
        r = r.copy()
        r[mask] += 2.0 * sign

    return rank01(r)

if expanded_accepted:
    selected_name = "expanded_nested_hill"
    selected_oof = expanded["oof"]
    selected_test = expanded["test"]
else:
    selected_name = "internal_nested_hill"
    selected_oof = internal_ref["oof"]
    selected_test = internal_ref["test"]

rules_oof = apply_rules(selected_oof, train)
rules_test = apply_rules(selected_test, test)

selected_auc = float(roc_auc_score(y, selected_oof))
rules_auc = float(roc_auc_score(y, rules_oof))

print("Selected:", selected_name)
print("Selected OOF:", selected_auc)
print("Rules OOF:", rules_auc)
print("Rules delta:", rules_auc - selected_auc)

Selected: internal_nested_hill
Selected OOF: 0.9461701638092593
Rules OOF: 0.9461792645831111
Rules delta: 9.10077385174457e-06


## 9. 保存最终文件

如果今晚只想提交一个，优先：

`submission_final_oof_only.csv`

如果还有一次额外提交机会，再试：

`submission_final_rules_experimental.csv`

In [10]:
oof_only_path = OUT_DIR / "submission_final_oof_only.csv"
rules_path = OUT_DIR / "submission_final_rules_experimental.csv"

save_submission(selected_test, oof_only_path)
save_submission(rules_test, rules_path)

pd.DataFrame({
    ID_COL: train[ID_COL].to_numpy(),
    "target": y.to_numpy(),
    "prediction": selected_oof,
}).to_csv(
    OUT_DIR / "oof_final_selected.csv",
    index=False,
)

all_auc.to_csv(OUT_DIR / "engine_auc.csv", index=False)
fold_compare.to_csv(OUT_DIR / "fold_comparison.csv", index=False)

if dropped:
    pd.DataFrame(dropped).to_csv(
        OUT_DIR / "dropped_near_duplicates.csv",
        index=False,
    )

selected_weights = (
    expanded["weights"]
    if expanded_accepted
    else internal_ref["weights"]
)

selected_weights.to_csv(
    OUT_DIR / "final_hill_weights.csv",
    header=["weight"],
)

report = {
    "selected": selected_name,
    "selected_oof_auc": selected_auc,
    "rules_oof_auc": rules_auc,
    "internal_oof_auc": internal_ref["auc"],
    "expanded_oof_auc": expanded["auc"],
    "expanded_delta": overall_delta,
    "expanded_wins": wins,
    "expanded_mean_fold_delta": mean_delta,
    "expanded_accepted": expanded_accepted,
    "external_engines_found": len(external_pairs),
    "eligible_engines": eligible_names,
    "kept_engines": kept,
    "output_oof_only": str(oof_only_path),
    "output_rules": str(rules_path),
}

(OUT_DIR / "final_report.json").write_text(
    json.dumps(report, indent=2),
    encoding="utf-8",
)

print("=" * 72)
print("FINAL RESULT")
print("=" * 72)
print("Selected:", selected_name)
print(f"OOF-only AUC: {selected_auc:.9f}")
print(f"Rules AUC   : {rules_auc:.9f}")
print()
print("Submit first:")
print(oof_only_path)
print()
print("Optional second submission:")
print(rules_path)
print()
print("Report:")
print(OUT_DIR / "final_report.json")

FINAL RESULT
Selected: internal_nested_hill
OOF-only AUC: 0.946170164
Rules AUC   : 0.946179265

Submit first:
/Users/yw/unsw/Kaggle/outputs/v11_final_20261001_031756_836748/submission_final_oof_only.csv

Optional second submission:
/Users/yw/unsw/Kaggle/outputs/v11_final_20261001_031756_836748/submission_final_rules_experimental.csv

Report:
/Users/yw/unsw/Kaggle/outputs/v11_final_20261001_031756_836748/final_report.json
